## 7 Прогноз для новых заявок и итоговые выводы

**Входы**

- Обученная модель и новые заявки без известного исхода

**Выходы**

- Оценки риска и классы заявок
- Выводы об итоговой модели и условия её применения

**Что делаем**

Получим прогноз для трёх придуманных заявок и подведём итоги по выбранной модели

#### Как получить прогноз

Передаём таблицу с 9 выбранными признаками, без `loan_status`, `loan_grade` и `loan_percent_income`. Для каждой строки получаем оценку риска и класс по выбранному порогу\
Применяем ту же обработку, что при обучении: заполняем пропуски, кодируем категории и масштабируем числовые признаки

In [ ]:
# include: _code/prepare_applications.py

In [ ]:
def predict_applications(applications, pipeline, threshold):
    prepared = prepare_applications(applications)
    scores = pipeline.predict_proba(prepared)[:, 1]
    return pd.DataFrame({
        'Оценка риска дефолта': scores,
        'Класс при выбранном пороге': (scores >= threshold).astype(int),
    }, index=applications.index)


new_applications = pd.DataFrame([
    [28, 65000, 4, 9000, 10.8, 6, 'RENT', 'EDUCATION', 'N'],
    [42, 38000, 8, 16000, 17.5, 12, 'MORTGAGE', 'DEBTCONSOLIDATION', 'Y'],
    [31, 92000, np.nan, 5000, np.nan, 8, 'OWN', 'PERSONAL', 'N'],
], columns=FEATURES, index=['Заявка 1', 'Заявка 2', 'Заявка 3'])
example_predictions = predict_applications(new_applications, selected_pipeline, selected_threshold)
display(new_applications)
display(example_predictions.round(4))


#### За чем следить при дальнейшем использовании

| Что проверять | Зачем |
| --- | --- |
| Поля, единицы измерения, пропуски и новые категории | Чтобы входные данные не меняли смысл |
| Изменение распределений по сравнению с train | Чтобы заметить, что новые заявки стали другими |
| Качество после появления фактических исходов | Чтобы понять, сохраняются ли точность, полнота и калибровка |
| Стоимость ошибок и качество для разных групп | Чтобы оценить последствия применения модели |
| Версии данных, обработки и порога | Чтобы знать, какой именно вариант дал прогноз |

Покажем проверку пропусков на трёх заявках, сравнив их с train. Для оценки изменений распределений понадобится поток реальных заявок

In [ ]:
monitoring_reference = pd.DataFrame({
    'Доля пропусков train': X_train.isna().mean(),
    'Доля пропусков примера': new_applications.isna().mean(),
})
display(monitoring_reference.round(4))


### Выводы об итоговой модели


In [ ]:
# include: _code/final_conclusions.py